# WHB ROI UMAP Highlight

This notebook follows the Allen WHB 10x snRNASeq tutorial metadata workflow and plots the WHB UMAP with only selected regions highlighted. The default selected regions are `Human A44-A45`, `Human A46`, `Human A32`, and `Human ACC`; all other cells are shown in grey.

Only metadata is downloaded: `cell_metadata` for UMAP coordinates and ROI labels, plus `region_of_interest_structure_map` for region descriptions.

## Imports

In [ ]:
from pathlib import Path
import json
import urllib.request

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D

## Configuration

In [ ]:
# Reuse the same cache location as the MerXen MapMyCells/WHB exploration work.
download_base = Path("/media/mathieubo/SSD2/MerXen/mapmycells/abc_whb")

# WHB stores A44 and A45 together as a single ROI label.
roi_labels = ["Human A44-A45", "Human A46", "Human A32", "Human ACC"]

# Colourblind-friendly-ish palette for the highlighted ROIs.
roi_palette = {
    "Human A44-A45": "#D55E00",
    "Human A46": "#0072B2",
    "Human A32": "#009E73",
    "Human ACC": "#CC79A7",
}

background_color = "#C9C9C9"
background_alpha = 0.22
background_subsample_every = 10
keep_all_highlighted_cells = True
background_point_size = 0.35
highlight_point_size = 1.2

save_figures = False
figure_dir = Path("../results/whb_roi_umap_highlight")

## Download/reuse WHB metadata

In [ ]:
MANIFEST_URL = "https://allen-brain-cell-atlas.s3.us-west-2.amazonaws.com/releases/20250531/manifest.json"
ABC_CACHE_DIR = download_base


def load_manifest(url: str = MANIFEST_URL) -> dict:
    with urllib.request.urlopen(url) as response:
        return json.loads(response.read().decode("utf-8"))


def manifest_file_info(manifest: dict, *keys: str) -> dict:
    node = manifest["file_listing"]
    for key in keys:
        node = node[key]
    return dict(node)


def ensure_manifest_file(info: dict, cache_dir: Path = ABC_CACHE_DIR) -> Path:
    output_path = cache_dir / info["relative_path"]
    expected_size = int(info.get("size", 0) or 0)
    if output_path.exists() and (not expected_size or output_path.stat().st_size == expected_size):
        return output_path

    output_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = output_path.with_name(output_path.name + ".tmp")
    print(f"Downloading {info['url']} -> {output_path}")
    with urllib.request.urlopen(info["url"]) as response, tmp_path.open("wb") as out:
        while True:
            chunk = response.read(16 * 1024 * 1024)
            if not chunk:
                break
            out.write(chunk)

    actual_size = tmp_path.stat().st_size
    if expected_size and actual_size != expected_size:
        tmp_path.unlink(missing_ok=True)
        raise RuntimeError(f"Downloaded size {actual_size:,} does not match expected {expected_size:,}")
    tmp_path.replace(output_path)
    return output_path


manifest = load_manifest()
cell_metadata_path = ensure_manifest_file(
    manifest_file_info(manifest, "WHB-10Xv3", "metadata", "cell_metadata", "files", "csv")
)
roi_map_path = ensure_manifest_file(
    manifest_file_info(manifest, "WHB-10Xv3", "metadata", "region_of_interest_structure_map", "files", "csv")
)

cell_metadata_path, roi_map_path

In [ ]:
print("WHB-10Xv3 metadata files:")
display(sorted(manifest["file_listing"]["WHB-10Xv3"]["metadata"].keys()))

In [ ]:
cell = pd.read_csv(cell_metadata_path, dtype={"cell_label": str})

if "cell_label" in cell.columns:
    cell = cell.set_index("cell_label")

roi = pd.read_csv(roi_map_path)
roi = roi.rename(columns={"color_hex_triplet": "allen_region_of_interest_color"})
roi = (
    roi.groupby("region_of_interest_label", as_index=True)
    .agg(
        structure_symbol=("structure_symbol", lambda values: "; ".join(pd.unique(values.dropna()))),
        structure_name=("structure_name", lambda values: "; ".join(pd.unique(values.dropna()))),
        allen_region_of_interest_color=("allen_region_of_interest_color", "first"),
    )
    .sort_index()
)

print(f"Number of cells = {len(cell):,}")
print(f"Number of ROI labels = {cell['region_of_interest_label'].nunique():,}")
cell.head()

## Check selected ROI labels and counts

In [ ]:
available_labels = set(cell["region_of_interest_label"].dropna().unique())
missing_labels = [label for label in roi_labels if label not in available_labels]

if missing_labels:
    raise ValueError(
        "These ROI labels were not found in WHB cell_metadata: "
        + ", ".join(missing_labels)
    )

missing_palette = [label for label in roi_labels if label not in roi_palette]
if missing_palette:
    raise ValueError(
        "These ROI labels need colours in roi_palette: "
        + ", ".join(missing_palette)
    )

roi_counts = (
    cell[cell["region_of_interest_label"].isin(roi_labels)]
    .pivot_table(
        index="region_of_interest_label",
        columns="feature_matrix_label",
        values="x",
        aggfunc="count",
        fill_value=0,
    )
    .reindex(roi_labels)
)
roi_counts["total"] = roi_counts.sum(axis=1)

roi_details = roi.loc[roi_labels, ["structure_symbol", "structure_name", "allen_region_of_interest_color"]]
roi_details = roi_details.join(roi_counts)
display(roi_details)

## Prepare plotting table

In [ ]:
plot_columns = ["feature_matrix_label", "x", "y", "region_of_interest_label"]
cell_plot = cell[plot_columns].copy()
cell_plot["is_highlight_roi"] = cell_plot["region_of_interest_label"].isin(roi_labels)
cell_plot["highlight_color"] = cell_plot["region_of_interest_label"].map(roi_palette)

feature_matrix_counts = cell_plot.groupby("feature_matrix_label")["x"].count().rename("n_cells")
display(feature_matrix_counts.to_frame())

## Plot helpers

In [ ]:
def split_for_plot(df, feature_matrix_label):
    feature_df = df[df["feature_matrix_label"] == feature_matrix_label]
    background = feature_df[~feature_df["is_highlight_roi"]]
    highlights = feature_df[feature_df["is_highlight_roi"]]

    if background_subsample_every and background_subsample_every > 1:
        background = background.iloc[::background_subsample_every]

    if not keep_all_highlighted_cells and background_subsample_every and background_subsample_every > 1:
        highlights = highlights.iloc[::background_subsample_every]

    return background, highlights


def plot_umap_highlight(df, feature_matrix_label, title, fig_width=8, fig_height=8):
    background, highlights = split_for_plot(df, feature_matrix_label)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.scatter(
        background["x"],
        background["y"],
        s=background_point_size,
        color=background_color,
        alpha=background_alpha,
        marker=".",
        linewidths=0,
        rasterized=True,
    )

    legend_handles = []
    for label in roi_labels:
        subset = highlights[highlights["region_of_interest_label"] == label]
        if subset.empty:
            continue

        color = roi_palette[label]
        ax.scatter(
            subset["x"],
            subset["y"],
            s=highlight_point_size,
            color=color,
            alpha=0.9,
            marker=".",
            linewidths=0,
            rasterized=True,
        )
        legend_handles.append(
            Line2D(
                [0],
                [0],
                marker="o",
                color="none",
                markerfacecolor=color,
                markeredgecolor="none",
                markersize=7,
                label=f"{label} (n={len(subset):,})",
            )
        )

    ax.axis("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
    ax.legend(
        handles=legend_handles,
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0,
    )
    fig.tight_layout()

    return fig, ax

## Neurons: selected ROIs over grey WHB background

In [ ]:
fig, ax = plot_umap_highlight(
    cell_plot,
    feature_matrix_label="WHB-10Xv3-Neurons",
    title="Neurons: A44/A45, A46, A32, and ACC highlighted",
)

if save_figures:
    figure_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(figure_dir / "whb_neurons_roi_highlight.png", dpi=300, bbox_inches="tight")

plt.show()

## Non-neurons: selected ROIs over grey WHB background

In [ ]:
fig, ax = plot_umap_highlight(
    cell_plot,
    feature_matrix_label="WHB-10Xv3-Nonneurons",
    title="Non-neurons: A44/A45, A46, A32, and ACC highlighted",
)

if save_figures:
    figure_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(figure_dir / "whb_non_neurons_roi_highlight.png", dpi=300, bbox_inches="tight")

plt.show()

## Optional: change the region set

To try a different region set, edit `roi_labels` and `roi_palette` in the configuration cell, then rerun the notebook from the validation cell onward. The notebook intentionally fails early if a requested WHB ROI label is not present.